# Episode 1: Reading files

## 0. Discovery: what is in this workspace?

Imagine your first day as a data engineer on a workspace you have never seen.
Nobody hands you a diagram. The only thing you know is that the company prefixes its catalogs with **`moustache_markt`**.

Everything else (catalogs, schemas, volumes, where the data actually lives) is **discovered by the code below**, not typed in by hand.

We query **`system.information_schema`** instead of running `SHOW ...` commands:
- it covers **every catalog** you have access to, in one place,
- it is a set of ordinary tables, so it can be filtered, joined and aggregated with SQL,
- the result is a DataFrame, so `collect()` turns it into Python lists we can reuse further down the notebook.

Unity Catalog only returns the objects **you have privileges on**. An empty result can mean "does not exist" or "you are not allowed to see it".

### 0.1 Catalogs

Two SQL details worth knowing:
- **`:prefix` is a named parameter marker.** `spark.sql(..., args={...})` substitutes the value safely, so we never glue values into SQL with f-strings (no quoting bugs, no SQL injection).
- **`startswith()` instead of `LIKE 'moustache_markt%'`.** In `LIKE` the underscore `_` means "any single character", so the pattern would also match e.g. `moustacheXmarkt`.

In [0]:
# The ONLY piece of knowledge we start with: the company's catalog prefix.
# Everything else in this notebook is derived from the workspace itself.
CATALOG_PREFIX = "moustache_markt"

# The same filter is reused in every discovery query below.
# The trailing "_" makes sure we match "moustache_markt_dev", but not "moustache_marktplace".
PREFIX_ARGS = {"prefix": CATALOG_PREFIX + "_"}

catalogs_df = spark.sql(
    """
    SELECT catalog_name, comment, catalog_owner, created, created_by
    FROM system.information_schema.catalogs
    WHERE startswith(catalog_name, :prefix)
    ORDER BY catalog_name
    """,
    args=PREFIX_ARGS,
)
display(catalogs_df)

# collect() brings the rows to the driver as a Python list of Row objects.
# Fine for a handful of metadata rows - never do it on a big table.
# Row fields are accessed by name: row.catalog_name.
CATALOGS = [row.catalog_name for row in catalogs_df.collect()]

# Fail fast: a typo in the prefix or missing permissions should stop the notebook HERE,
# not show up as mysteriously empty results three cells later.
assert CATALOGS, f"No catalogs starting with '{CATALOG_PREFIX}_' - typo in the prefix or no USE CATALOG privilege?"
print(f"Found {len(CATALOGS)} catalog(s): {CATALOGS}")

: 

### 0.2 Schemas (and are there any tables in them?)

For every schema we also count its tables with a `LEFT JOIN` to `information_schema.tables`.
A plain `JOIN` would drop schemas that have **no** tables, and those are exactly the ones we want to spot on a fresh workspace.

- `information_schema` is skipped: Unity Catalog creates it automatically in every catalog.
- `default` is also created automatically with every catalog. It is kept, so you can see it is empty.

In [0]:
from pyspark.sql import functions as F

schemas_df = spark.sql(
    """
    SELECT s.catalog_name,
           s.schema_name,
           count(t.table_name) AS tables,   -- count(column) skips NULLs -> 0 for schemas without tables
           s.comment
    FROM system.information_schema.schemata AS s
    LEFT JOIN system.information_schema.tables AS t
           ON t.table_catalog = s.catalog_name
          AND t.table_schema  = s.schema_name
    WHERE startswith(s.catalog_name, :prefix)
      AND s.schema_name <> 'information_schema'
    GROUP BY ALL                             -- group by every non-aggregated column in SELECT
    ORDER BY s.catalog_name, s.schema_name
    """,
    args=PREFIX_ARGS,
)
display(schemas_df)

# One row per catalog: how many schemas, which ones and how many tables in total.
# array_sort makes the list order stable, so catalogs with the same structure look identical
# (collect_list on its own returns elements in whatever order the rows happened to arrive).
display(
    schemas_df.groupBy("catalog_name")
    .agg(
        F.count("*").alias("schemas"),
        F.sum("tables").alias("tables"),
        F.array_sort(F.collect_list("schema_name")).alias("schema_names"),
    )
    .orderBy("catalog_name")
)

SCHEMAS = [(row.catalog_name, row.schema_name) for row in schemas_df.collect()]
TOTAL_TABLES = sum(row.tables for row in schemas_df.collect())
print(f"{len(SCHEMAS)} schema(s), {TOTAL_TABLES} table(s) in total")
if TOTAL_TABLES == 0:
    print("No tables anywhere - if there is any data, it must be sitting in volumes as files.")

### 0.3 Volumes: where are the files?

A **volume** is a Unity Catalog object for **files** (as opposed to tables).
Every volume has a fixed path:

```
/Volumes/<catalog>/<schema>/<volume>/...
```

For each volume we build that path and peek at its **top-level entries only** (no recursion yet, that comes in the next step).
This tells us which volumes actually hold data and how each one is organised.

**Why `dbutils.fs.ls` and not `os.listdir`?** `/Volumes` is a mount that exists **only on Databricks compute**.
In a workspace notebook `os.listdir("/Volumes/...")` works, but run the same cell from VS Code (Databricks Connect)
and Python looks for `/Volumes` on **your laptop**: `FileNotFoundError`, even though the volume exists.
`dbutils.fs.ls` goes through the Databricks API, so it works in both places.

In [0]:
volumes_df = spark.sql(
    """
    SELECT volume_catalog, volume_schema, volume_name, volume_type, comment
    FROM system.information_schema.volumes
    WHERE startswith(volume_catalog, :prefix)
    ORDER BY volume_catalog, volume_schema, volume_name
    """,
    args=PREFIX_ARGS,
)
display(volumes_df)

# One dict per volume: its full name, its /Volumes path and its top-level entries.
VOLUMES = []
for row in volumes_df.collect():
    path = f"/Volumes/{row.volume_catalog}/{row.volume_schema}/{row.volume_name}"
    # dbutils.fs.ls = only the direct children (files and folders), not the whole tree.
    # On Databricks a folder name ends with "/" ("product/"), in Databricks Connect it does not ("product").
    # rstrip("/") gives the same result in both places.
    # sorted() because the listing comes back in no particular order.
    entries = sorted(f.name.rstrip("/") for f in dbutils.fs.ls(path))
    VOLUMES.append({
        "full_name": f"{row.volume_catalog}.{row.volume_schema}.{row.volume_name}",
        "path": path,
        "entries": entries,
    })

# Volumes that contain anything at all - the candidates for the rest of this episode.
DATA_VOLUMES = [v for v in VOLUMES if v["entries"]]

# A quick text tree: one line per volume, empty ones clearly marked.
for v in VOLUMES:
    content = ", ".join(v["entries"]) if v["entries"] else "(empty)"
    print(f"{v['full_name']:<45} {content}")

print(f"\n{len(DATA_VOLUMES)} of {len(VOLUMES)} volume(s) contain data:")
for v in DATA_VOLUMES:
    print(f"  {v['path']}")

In [0]:
dbutils.fs.ls('/Volumes/moustache_markt_sources/landing/erp')

In [0]:
# variables

FORMATS = {"erp": "parquet", "crm": "jsonl", "ecom": "json_array", "pos": "csv"}

In [0]:
#   1. Pobiera listę volume'ów z information_schema.
#   2. Listuje subfoldery pierwszego poziomu w każdym volume.
#   3. Skanuje równolegle każdy subfolder i zbiera unikalne rozszerzenia.

#   Wszystko działa przez os.scandir, więc nadaje się na serverless.

import os
from collections import defaultdict, Counter
from concurrent.futures import ThreadPoolExecutor

CATALOG = "moustache_markt_sources"

def pathlib_ext(name):
    return "." + name.split(".")[-1] if "." in name else "(no extension)"

def ls(path):
    with os.scandir(path) as it:
        return [(e.path, e.name, e.is_dir()) for e in it]

# 1. Lista volume'ów w katalogu
volumes = spark.sql(f"""
    SELECT volume_schema, volume_name
    FROM {CATALOG}.information_schema.volumes
    ORDER BY volume_schema, volume_name
""")
display(volumes)

samples = {}   # (volume, folder, ext) -> ścieżka pierwszego pliku (alfabetycznie)

# 2. Subfoldery pierwszego poziomu w każdym volume (np. pos/receipt_line)
stats = defaultdict(Counter)   # (volume, folder) -> Counter rozszerzeń
tasks = []                     # ((volume, folder), ścieżka)
for v in volumes.collect():
    root = f"/Volumes/{CATALOG}/{v.volume_schema}/{v.volume_name}"
    for path, name, is_dir in ls(root):
        if is_dir:
            key = (v.volume_name, name)
            stats[key]  # puste foldery też mają się pokazać w wyniku
            tasks.append((key, path))
        else:
            stats[(v.volume_name, "(root)")][pathlib_ext(name)] += 1

# 3. Rekurencyjny skan każdego subfolderu, równolegle, poziom po poziomie
def scan(tasks, workers=32):
    level = tasks
    with ThreadPoolExecutor(max_workers=workers) as pool:
        while level:
            next_level = []
            results = pool.map(ls, [path for _, path in level])
            for (key, _), entries in zip(level, results):
                for path, name, is_dir in entries:
                    if is_dir:
                        next_level.append((key, path))
                    else:
                        stats[key][pathlib_ext(name)] += 1
            level = next_level

scan(tasks)

# Wynik: unikalne rozszerzenia per volume/folder
rows = [
    (vol, folder, sorted(c), dict(c), sum(c.values()))
    for (vol, folder), c in sorted(stats.items())
]
result = spark.createDataFrame(
    rows,
    "volume string, folder string, extensions array<string>, "
    "ext_counts map<string,bigint>, file_count bigint",
)
display(result)

#   Wynik wygląda mniej więcej tak:

#   ┌────────┬──────────────┬───────────────────┬─────────────────────────────┬────────────┐
#   │ volume │    folder    │    extensions     │         ext_counts          │ file_count │
#   ├────────┼──────────────┼───────────────────┼─────────────────────────────┼────────────┤
#   │ erp    │ category     │ [.csv]            │ {.csv → 12}                 │ 12         │
#   ├────────┼──────────────┼───────────────────┼─────────────────────────────┼────────────┤
#   │ pos    │ receipt_line │ [.json, .parquet] │ {.json → 3, .parquet → 480} │ 483        │
#   └────────┴──────────────┴───────────────────┴─────────────────────────────┴────────────┘

#   Kilka uwag:

#   - Pliki leżące bezpośrednio w volume, a nie w subfolderze, trafiają do wiersza (root).
#   - Puste subfoldery też się pokazują, z pustą listą rozszerzeń. Dzięki temu widać, że folder istnieje.
#   - Jeśli w katalogu są inne schematy niż landing i chcesz brać pod uwagę tylko landing, dopisz do
#     zapytania WHERE volume_schema = 'landing'. Jeśli dwa schematy mają volume o tej samej nazwie, dodaj
#     schemat do klucza, inaczej ich statystyki się zsumują.
#   - Kolumna ext_counts to dodatek. Jeśli potrzebujesz tylko unikalnych rozszerzeń, wystarczy kolumna
#     extensions.

In [0]:
# checking data
# One sample file from every folder, read with the right reader and shown as a table.


def read_csv(path):
    # POS: "|" as the delimiter, Windows encoding (cp1252), first line = column names
    return spark.read.option("header", True).option("sep", "|").option("encoding", "cp1252").csv(path)


def read_json(path):
    # e-commerce: the whole file is one JSON array, so Spark has to read it as a single document
    return spark.read.option("multiLine", True).json(path)


def read_jsonl(path):
    # CRM: one JSON object per line - the default for spark.read.json
    return spark.read.json(path)


def read_parquet(path):
    # ERP: the schema is stored inside the file, no options needed
    return spark.read.parquet(path)


READERS = {"csv": read_csv, "json": read_json, "jsonl": read_jsonl, "parquet": read_parquet}

def read_all_files():
    for volume in DATA_VOLUMES:
        for folder in volume["entries"]:
            # folder -> first load_date=... subfolder -> first file in it
            first_day = dbutils.fs.ls(f"{volume['path']}/{folder}")[0].path
            sample = dbutils.fs.ls(first_day)[0].path
            extension = sample.split(".")[-1]

            print(f"{folder}: {sample}")
            display(READERS[extension](sample).limit(10))

read_all_files()